# Prime Numbers Lab — Notebook 14: Residual Scaling Law

**Repo:** `github.com/thinkthoughts/prime-numbers-lab`  
**Notebook purpose:** test whether normalized prime-gap residuals decay with scale according to a measurable scaling law.

**Core frame:**  
Constraint → structure remains under constraint; drift marks invalid assignments; structure may remain recoverable from partial observation.

Notebook 13 measured the residual:

\[
\Delta(z,x)=f_{\mathrm{emp}}(z,x)-e^{-z}
\]

Notebook 14 asks whether the residual magnitude follows:

\[
\|\Delta(z,x)\| \sim C(\log x)^{-\alpha}.
\]

## 0. Setup

This notebook follows the established `prime-numbers-lab` template:

1. define one constraint  
2. generate one dataset  
3. measure what remains under constraint  
4. visualize drift / retention / recoverability  
5. export figures, data, notes, and TeX  
6. package results into a root-level export zip

In [ ]:

# Standard library
from pathlib import Path
import json
import math
import zipfile

# Data / compute
import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt

# Notebook identity
NOTEBOOK_ID = "14_residual_scaling_law"
NOTEBOOK_TITLE = "Residual Scaling Law"
REPO_NAME = "prime-numbers-lab"
NOTEBOOK_NUM = NOTEBOOK_ID.split("_")[0]

# Output directories
OUT = Path(NOTEBOOK_ID)
FIG_DIR = OUT / "figures"
DATA_DIR = OUT / "data"
DOCS_DIR = OUT / "docs"
TEX_DIR = OUT / "tex"

for d in [FIG_DIR, DATA_DIR, DOCS_DIR, TEX_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {OUT.resolve()}")

## 1. Premise

Notebook 13 showed that normalized prime-gap residuals shrink with scale.

Notebook 14 measures whether that shrinkage follows a log-scale power law:

\[
\|\Delta(z,x)\| \sim C(\log x)^{-\alpha}.
\]

**Short vocabulary:**

- **Remains under constraint / persists:** exponential envelope remains visible after normalization.
- **Drift:** residual deviation from the exponential baseline.
- **Recoverability:** whether residual decay can be summarized by a fitted scaling law.

Core question:

> Is the residual decay visual only, or does it follow a measurable scale law?

## 2. Constraint definition

The normalized prime gap is:

\[
z_n = \frac{p_{n+1}-p_n}{\log p_n}
\]

The exponential reference distribution is:

\[
f_0(z)=e^{-z}
\]

The residual is:

\[
\Delta(z,x)=f_{\mathrm{emp}}(z,x)-f_0(z)
\]

The scaling model is:

\[
\|\Delta(z,x)\| \sim C(\log x)^{-\alpha}
\]

Taking logarithms:

\[
\log \|\Delta(z,x)\| = \log C - \alpha \log\log x
\]

In [ ]:

# Notebook-specific parameters

N_MAX = 2_000_000
RANDOM_SEED = 9423

Z_MAX = 6.0
BIN_COUNT = 90

WINDOW_COUNT = 18
MIN_WINDOW_GAPS = 200

rng = np.random.default_rng(RANDOM_SEED)

params = {
    "N_MAX": N_MAX,
    "RANDOM_SEED": RANDOM_SEED,
    "Z_MAX": Z_MAX,
    "BIN_COUNT": BIN_COUNT,
    "WINDOW_COUNT": WINDOW_COUNT,
    "MIN_WINDOW_GAPS": MIN_WINDOW_GAPS,
    "NOTEBOOK_ID": NOTEBOOK_ID,
    "NOTEBOOK_TITLE": NOTEBOOK_TITLE,
}

params

## 3. Data generation

Generate primes up to \(N_{\max}\), then compute gaps and normalized gaps.

In [ ]:

def generate_primes(n_max: int) -> np.ndarray:
    if n_max < 2:
        return np.array([], dtype=int)

    sieve = np.ones(n_max + 1, dtype=bool)
    sieve[:2] = False

    for i in range(2, int(math.sqrt(n_max)) + 1):
        if sieve[i]:
            sieve[i*i:n_max+1:i] = False

    return np.nonzero(sieve)[0].astype(int)

primes = generate_primes(N_MAX)
gaps = np.diff(primes)
anchors = primes[:-1]
normalized_gaps = gaps / np.log(anchors)

summary = {
    "n_max": int(N_MAX),
    "prime_count": int(len(primes)),
    "gap_count": int(len(gaps)),
    "first_primes": primes[:10].tolist(),
    "last_primes": primes[-10:].tolist(),
    "mean_gap": float(np.mean(gaps)),
    "mean_normalized_gap": float(np.mean(normalized_gaps)),
    "std_normalized_gap": float(np.std(normalized_gaps)),
}

summary

## 4. Window construction

Use logarithmically spaced windows to measure scale-dependent residuals.

Each window collects gaps whose left prime \(p_n\) lies inside the window.

In [ ]:

raw_edges = np.unique(np.logspace(np.log10(100), np.log10(N_MAX), WINDOW_COUNT + 1).astype(int))
raw_edges[0] = 2
raw_edges[-1] = N_MAX

window_rows = []
window_gap_arrays = []

for idx, (left, right) in enumerate(zip(raw_edges[:-1], raw_edges[1:]), start=1):
    mask = (anchors >= left) & (anchors < right)
    z = normalized_gaps[mask]
    g = gaps[mask]
    x = anchors[mask]

    if len(z) < MIN_WINDOW_GAPS:
        continue

    row = {
        "window_index": idx,
        "left": int(left),
        "right": int(right),
        "midpoint": float(math.sqrt(left * right)),
        "gap_count": int(len(z)),
        "mean_normalized_gap": float(np.mean(z)),
        "std_normalized_gap": float(np.std(z)),
        "median_normalized_gap": float(np.median(z)),
        "q75_normalized_gap": float(np.quantile(z, 0.75)),
        "q90_normalized_gap": float(np.quantile(z, 0.90)),
        "q95_normalized_gap": float(np.quantile(z, 0.95)),
    }

    window_rows.append(row)
    window_gap_arrays.append({
        "window_index": idx,
        "left": int(left),
        "right": int(right),
        "midpoint": float(math.sqrt(left * right)),
        "z": z,
        "gaps": g,
        "anchors": x,
    })

windows_df = pd.DataFrame(window_rows)

windows_df.head(), windows_df.tail()

## 5. Measurement

Estimate empirical PDFs and compare each to the exponential baseline.

Outputs:

- residual \(L_1\)
- residual \(L_2\)
- Jensen–Shannon divergence
- positive / negative residual mass
- tail bias

In [ ]:

def safe_normalize_pdf(pdf: np.ndarray, bin_width: float) -> np.ndarray:
    total = float(np.sum(pdf) * bin_width)
    if total <= 0:
        return pdf
    return pdf / total

def js_divergence_discrete(p_density: np.ndarray, q_density: np.ndarray, bin_width: float) -> float:
    p = np.maximum(p_density * bin_width, 1e-15)
    q = np.maximum(q_density * bin_width, 1e-15)

    p = p / p.sum()
    q = q / q.sum()
    m = 0.5 * (p + q)

    kl_pm = np.sum(p * np.log(p / m))
    kl_qm = np.sum(q * np.log(q / m))

    return float(0.5 * (kl_pm + kl_qm))

bins = np.linspace(0, Z_MAX, BIN_COUNT + 1)
centers = 0.5 * (bins[:-1] + bins[1:])
bin_width = float(bins[1] - bins[0])

exp_pdf = np.exp(-centers)
exp_pdf = safe_normalize_pdf(exp_pdf, bin_width)

residual_rows = []
residual_grid_rows = []

for item in window_gap_arrays:
    idx = item["window_index"]
    z = item["z"]

    hist, _ = np.histogram(z, bins=bins, density=True)
    hist = safe_normalize_pdf(hist, bin_width)

    delta = hist - exp_pdf

    l1 = float(np.sum(np.abs(delta)) * bin_width)
    l2 = float(np.sqrt(np.sum(delta**2) * bin_width))
    signed_mean = float(np.sum(delta) * bin_width)

    positive_mass = float(np.sum(np.maximum(delta, 0)) * bin_width)
    negative_mass = float(np.sum(np.minimum(delta, 0)) * bin_width)

    js = js_divergence_discrete(hist, exp_pdf, bin_width)

    tail_mask = centers >= 3.0
    tail_bias = float(np.sum(delta[tail_mask]) * bin_width)

    residual_rows.append({
        "window_index": idx,
        "left": item["left"],
        "right": item["right"],
        "midpoint": item["midpoint"],
        "gap_count": int(len(z)),
        "residual_l1": l1,
        "residual_l2": l2,
        "residual_signed_mean": signed_mean,
        "positive_residual_mass": positive_mass,
        "negative_residual_mass": negative_mass,
        "js_divergence": js,
        "tail_bias_z_ge_3": tail_bias,
    })

    for c, h, e, d in zip(centers, hist, exp_pdf, delta):
        residual_grid_rows.append({
            "window_index": idx,
            "midpoint": item["midpoint"],
            "z_center": float(c),
            "empirical_pdf": float(h),
            "exp_pdf": float(e),
            "delta": float(d),
        })

residual_metrics_df = pd.DataFrame(residual_rows)
residual_grid_df = pd.DataFrame(residual_grid_rows)

measurement = {
    "mean_residual_l1": float(residual_metrics_df["residual_l1"].mean()),
    "final_residual_l1": float(residual_metrics_df["residual_l1"].iloc[-1]),
    "mean_residual_l2": float(residual_metrics_df["residual_l2"].mean()),
    "final_residual_l2": float(residual_metrics_df["residual_l2"].iloc[-1]),
    "mean_js_divergence": float(residual_metrics_df["js_divergence"].mean()),
    "final_js_divergence": float(residual_metrics_df["js_divergence"].iloc[-1]),
    "mean_tail_bias_z_ge_3": float(residual_metrics_df["tail_bias_z_ge_3"].mean()),
    "final_tail_bias_z_ge_3": float(residual_metrics_df["tail_bias_z_ge_3"].iloc[-1]),
    "window_count_used": int(len(residual_metrics_df)),
}

measurement

## 6. Scaling-law fit

Fit:

\[
\log \|\Delta\| = \log C - \alpha \log\log x
\]

for \(L_1\), \(L_2\), and JS divergence.

In [ ]:

def fit_scaling_law(df: pd.DataFrame, metric_col: str):
    fit_df = df[["midpoint", metric_col]].replace([np.inf, -np.inf], np.nan).dropna()
    fit_df = fit_df[fit_df[metric_col] > 0].copy()

    X = np.log(np.log(fit_df["midpoint"].values))
    Y = np.log(fit_df[metric_col].values)

    slope, intercept = np.polyfit(X, Y, 1)
    yhat = slope * X + intercept

    ss_res = float(np.sum((Y - yhat) ** 2))
    ss_tot = float(np.sum((Y - np.mean(Y)) ** 2))
    r2 = float(1 - ss_res / ss_tot) if ss_tot > 0 else float("nan")

    alpha = float(-slope)
    C = float(np.exp(intercept))

    return {
        "metric": metric_col,
        "alpha": alpha,
        "C": C,
        "slope": float(slope),
        "intercept": float(intercept),
        "r2": r2,
        "n_fit": int(len(fit_df)),
    }, fit_df, X, Y, yhat

fit_results = []
fit_cache = {}

for metric_col in ["residual_l1", "residual_l2", "js_divergence"]:
    result, fit_df, X, Y, yhat = fit_scaling_law(residual_metrics_df, metric_col)
    fit_results.append(result)
    fit_cache[metric_col] = {
        "fit_df": fit_df,
        "X": X,
        "Y": Y,
        "yhat": yhat,
        "result": result,
    }

scaling_fit_df = pd.DataFrame(fit_results)

scaling_fit_df

## 7. CGCS score

The score measures residual scaling strength.

Working definition:

\[
CGCS_{\mathrm{scale}}=
\frac{1}{1+\overline{\|\Delta\|_1}+\overline{JS}+|\overline{\mathrm{fit\ residual}}|}
\]

where fit residuals are measured in log space.

In [ ]:

fit_residual_abs = []

for metric_col, cache in fit_cache.items():
    fit_residual_abs.extend(np.abs(cache["Y"] - cache["yhat"]).tolist())

mean_abs_fit_residual = float(np.mean(fit_residual_abs))

cgcs_score = 1.0 / (
    1.0
    + measurement["mean_residual_l1"]
    + measurement["mean_js_divergence"]
    + mean_abs_fit_residual
)

cgcs = {
    "score": float(cgcs_score),
    "definition": "1 / (1 + mean residual L1 + mean JS divergence + mean abs log-fit residual)",
    "interpretation": "Closer to 1 indicates smaller residuals and stronger scaling-law agreement.",
    "mean_abs_log_fit_residual": mean_abs_fit_residual,
}

cgcs

## 8. Visualization

Notebook 14 produces scaling-law figures:

1. residual norm vs scale  
2. L1 scaling fit  
3. L2 scaling fit  
4. JS scaling fit  
5. alpha comparison  
6. fit residuals  
7. residual heatmap  
8. positive / negative mass  
9. convergence score  
10. tail residual bias  
11. final window PDF comparison

### Figure 1 — residual norm vs scale

In [ ]:

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(residual_metrics_df["midpoint"], residual_metrics_df["residual_l1"], marker="o", label="L1 residual")
ax.plot(residual_metrics_df["midpoint"], residual_metrics_df["residual_l2"], marker="o", label="L2 residual")
ax.set_xscale("log")
ax.set_title("Residual norm vs scale")
ax.set_xlabel("window midpoint x")
ax.set_ylabel("residual norm")
ax.legend()
ax.grid(True, alpha=0.3)

fig1_path = FIG_DIR / f"{NOTEBOOK_NUM}_residual_norm_vs_scale.png"
fig.savefig(fig1_path, dpi=180, bbox_inches="tight")
plt.show()

fig1_path

### Figure 2 — L1 scaling fit

In [ ]:

cache = fit_cache["residual_l1"]
fit_df = cache["fit_df"]
result = cache["result"]

x_grid = np.geomspace(fit_df["midpoint"].min(), fit_df["midpoint"].max(), 200)
model = result["C"] * (np.log(x_grid) ** (-result["alpha"]))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(fit_df["midpoint"], fit_df["residual_l1"], marker="o", linestyle="", label="observed L1")
ax.plot(x_grid, model, label=f"fit alpha={result['alpha']:.3f}, R2={result['r2']:.3f}")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("L1 residual scaling fit")
ax.set_xlabel("window midpoint x")
ax.set_ylabel("L1 residual")
ax.legend()
ax.grid(True, which="both", alpha=0.3)

fig2_path = FIG_DIR / f"{NOTEBOOK_NUM}_l1_scaling_fit.png"
fig.savefig(fig2_path, dpi=180, bbox_inches="tight")
plt.show()

fig2_path

### Figure 3 — L2 scaling fit

In [ ]:

cache = fit_cache["residual_l2"]
fit_df = cache["fit_df"]
result = cache["result"]

x_grid = np.geomspace(fit_df["midpoint"].min(), fit_df["midpoint"].max(), 200)
model = result["C"] * (np.log(x_grid) ** (-result["alpha"]))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(fit_df["midpoint"], fit_df["residual_l2"], marker="o", linestyle="", label="observed L2")
ax.plot(x_grid, model, label=f"fit alpha={result['alpha']:.3f}, R2={result['r2']:.3f}")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("L2 residual scaling fit")
ax.set_xlabel("window midpoint x")
ax.set_ylabel("L2 residual")
ax.legend()
ax.grid(True, which="both", alpha=0.3)

fig3_path = FIG_DIR / f"{NOTEBOOK_NUM}_l2_scaling_fit.png"
fig.savefig(fig3_path, dpi=180, bbox_inches="tight")
plt.show()

fig3_path

### Figure 4 — JS scaling fit

In [ ]:

cache = fit_cache["js_divergence"]
fit_df = cache["fit_df"]
result = cache["result"]

x_grid = np.geomspace(fit_df["midpoint"].min(), fit_df["midpoint"].max(), 200)
model = result["C"] * (np.log(x_grid) ** (-result["alpha"]))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(fit_df["midpoint"], fit_df["js_divergence"], marker="o", linestyle="", label="observed JS")
ax.plot(x_grid, model, label=f"fit alpha={result['alpha']:.3f}, R2={result['r2']:.3f}")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("JS divergence scaling fit")
ax.set_xlabel("window midpoint x")
ax.set_ylabel("JS divergence")
ax.legend()
ax.grid(True, which="both", alpha=0.3)

fig4_path = FIG_DIR / f"{NOTEBOOK_NUM}_js_scaling_fit.png"
fig.savefig(fig4_path, dpi=180, bbox_inches="tight")
plt.show()

fig4_path

### Figure 5 — alpha comparison

In [ ]:

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(scaling_fit_df["metric"], scaling_fit_df["alpha"])
ax.set_title("Scaling exponent alpha comparison")
ax.set_xlabel("metric")
ax.set_ylabel("alpha")
ax.tick_params(axis="x", rotation=20)
ax.grid(True, axis="y", alpha=0.3)

fig5_path = FIG_DIR / f"{NOTEBOOK_NUM}_alpha_comparison.png"
fig.savefig(fig5_path, dpi=180, bbox_inches="tight")
plt.show()

fig5_path

### Figure 6 — fit residuals

In [ ]:

fig, ax = plt.subplots(figsize=(8, 5))

for metric_col, cache in fit_cache.items():
    fit_df = cache["fit_df"]
    residual = cache["Y"] - cache["yhat"]
    ax.plot(fit_df["midpoint"], residual, marker="o", label=metric_col)

ax.axhline(0, linestyle="--", linewidth=1)
ax.set_xscale("log")
ax.set_title("Scaling-law fit residuals")
ax.set_xlabel("window midpoint x")
ax.set_ylabel("log-fit residual")
ax.legend()
ax.grid(True, alpha=0.3)

fig6_path = FIG_DIR / f"{NOTEBOOK_NUM}_fit_residuals.png"
fig.savefig(fig6_path, dpi=180, bbox_inches="tight")
plt.show()

fig6_path

### Figure 7 — residual heatmap

In [ ]:

pivot_delta = residual_grid_df.pivot(index="window_index", columns="z_center", values="delta")

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(
    pivot_delta.values,
    aspect="auto",
    origin="lower",
    interpolation="nearest",
    extent=[centers.min(), centers.max(), 0, len(pivot_delta.index)],
)
ax.axvline(1.0, linestyle="--", linewidth=1)
ax.axvline(3.0, linestyle="--", linewidth=1)
ax.set_title("Residual heatmap across scale")
ax.set_xlabel("normalized gap z")
ax.set_ylabel("window index")
fig.colorbar(im, ax=ax, label="Delta(z, x)")

fig7_path = FIG_DIR / f"{NOTEBOOK_NUM}_residual_heatmap_across_scale.png"
fig.savefig(fig7_path, dpi=180, bbox_inches="tight")
plt.show()

fig7_path

### Figure 8 — positive and negative residual mass

In [ ]:

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(residual_metrics_df["midpoint"], residual_metrics_df["positive_residual_mass"], marker="o", label="positive residual mass")
ax.plot(residual_metrics_df["midpoint"], -residual_metrics_df["negative_residual_mass"], marker="o", label="negative residual mass magnitude")
ax.set_xscale("log")
ax.set_title("Positive / negative residual mass")
ax.set_xlabel("window midpoint x")
ax.set_ylabel("mass")
ax.legend()
ax.grid(True, alpha=0.3)

fig8_path = FIG_DIR / f"{NOTEBOOK_NUM}_positive_negative_residual_mass.png"
fig.savefig(fig8_path, dpi=180, bbox_inches="tight")
plt.show()

fig8_path

### Figure 9 — distributional convergence score

In [ ]:

residual_metrics_df["distributional_convergence_score"] = 1.0 / (
    1.0
    + residual_metrics_df["residual_l1"]
    + residual_metrics_df["residual_l2"]
    + residual_metrics_df["js_divergence"]
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(residual_metrics_df["midpoint"], residual_metrics_df["distributional_convergence_score"], marker="o")
ax.set_xscale("log")
ax.set_title("Distributional convergence score")
ax.set_xlabel("window midpoint x")
ax.set_ylabel("score")
ax.grid(True, alpha=0.3)

fig9_path = FIG_DIR / f"{NOTEBOOK_NUM}_distributional_convergence_score.png"
fig.savefig(fig9_path, dpi=180, bbox_inches="tight")
plt.show()

fig9_path

### Figure 10 — tail residual bias

In [ ]:

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(residual_metrics_df["midpoint"], residual_metrics_df["tail_bias_z_ge_3"], marker="o")
ax.axhline(0, linestyle="--", linewidth=1)
ax.set_xscale("log")
ax.set_title("Tail residual bias for z >= 3")
ax.set_xlabel("window midpoint x")
ax.set_ylabel("tail bias")
ax.grid(True, alpha=0.3)

fig10_path = FIG_DIR / f"{NOTEBOOK_NUM}_tail_residual_bias_z_ge_3.png"
fig.savefig(fig10_path, dpi=180, bbox_inches="tight")
plt.show()

fig10_path

### Figure 11 — final window PDF comparison

In [ ]:

final_idx = int(residual_metrics_df["window_index"].iloc[-1])
final_sub = residual_grid_df[residual_grid_df["window_index"] == final_idx].copy()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(final_sub["z_center"], final_sub["empirical_pdf"], marker="o", markersize=3, label="empirical PDF")
ax.plot(final_sub["z_center"], final_sub["exp_pdf"], linewidth=2, label="Exp(1)")
ax.set_title("Final window PDF vs Exp(1)")
ax.set_xlabel("normalized gap z")
ax.set_ylabel("density")
ax.legend()
ax.grid(True, alpha=0.3)

fig11_path = FIG_DIR / f"{NOTEBOOK_NUM}_final_window_pdf_vs_exp1.png"
fig.savefig(fig11_path, dpi=180, bbox_inches="tight")
plt.show()

fig11_path

## 9. Interpretation

Use a short, consistent structure:

1. **What remains under constraint?**  
2. **What drifts?**  
3. **What appears recoverable?**  
4. **What should not be overclaimed?**

In [ ]:

best_alpha_l1 = float(scaling_fit_df.loc[scaling_fit_df["metric"] == "residual_l1", "alpha"].iloc[0])
best_alpha_l2 = float(scaling_fit_df.loc[scaling_fit_df["metric"] == "residual_l2", "alpha"].iloc[0])
best_alpha_js = float(scaling_fit_df.loc[scaling_fit_df["metric"] == "js_divergence", "alpha"].iloc[0])

interpretation = f'''
# {NOTEBOOK_TITLE}

## Constraint result

This notebook tests whether normalized prime-gap residuals decay by a log-scale law.

The fitted model is:

$$
\\|\\Delta(z,x)\\| \\sim C(\\log x)^{{-\\alpha}}.
$$

## Remains under constraint

The exponential envelope remains visible after normalization, and residual metrics can be fit by a compact scale model.

Fitted exponents:

- L1 alpha = {best_alpha_l1:.6f}
- L2 alpha = {best_alpha_l2:.6f}
- JS alpha = {best_alpha_js:.6f}

## Drift

Drift is measured by residual L1, residual L2, JS divergence, positive/negative residual mass, and tail bias.

## Recoverability

The residual trend is recoverable as a finite-scale diagnostic:

- residual norm vs scale
- log-scale power-law fit
- alpha comparison
- residual heatmap
- fit residuals

## CGCS score

The scaling agreement score is:

$$
CGCS_{{scale}} =
\\frac{{1}}{{1+\\overline{{\\|\\Delta\\|_1}}+\\overline{{JS}}+\\overline{{|r_{{fit}}|}}}}.
$$

Measured score:

$$
CGCS_{{scale}} = {cgcs_score:.6f}.
$$

## Caution

This notebook does not prove an asymptotic theorem.

It measures finite residual scaling relative to the exponential heuristic.
'''.strip()

print(interpretation)

## 10. Export data, notes, figures index, math, and TeX

This block writes reusable artifacts:

- CSV summaries
- JSON metadata
- Markdown interpretation with embedded figure links
- Markdown design notes
- TeX results snippet
- standalone TeX math notes

In [ ]:

summary_df = pd.DataFrame([{
    **params,
    **summary,
    **measurement,
    "cgcs_score": cgcs["score"],
    "cgcs_definition": cgcs["definition"],
    "mean_abs_log_fit_residual": cgcs["mean_abs_log_fit_residual"],
}])

summary_path = DATA_DIR / f"{NOTEBOOK_NUM}_summary.csv"
windows_path = DATA_DIR / f"{NOTEBOOK_NUM}_windows.csv"
residual_metrics_path = DATA_DIR / f"{NOTEBOOK_NUM}_residual_metrics.csv"
residual_grid_path = DATA_DIR / f"{NOTEBOOK_NUM}_residual_grid.csv"
scaling_fit_path = DATA_DIR / f"{NOTEBOOK_NUM}_scaling_fit_summary.csv"
metadata_path = DATA_DIR / f"{NOTEBOOK_NUM}_metadata.json"

interpretation_md_path = DOCS_DIR / f"{NOTEBOOK_NUM}_interpretation.md"
design_notes_md_path = DOCS_DIR / f"{NOTEBOOK_NUM}_design_notes.md"

summary_tex_path = TEX_DIR / f"{NOTEBOOK_NUM}_summary_snippet.tex"
math_tex_path = TEX_DIR / f"{NOTEBOOK_NUM}_math_notes.tex"

summary_df.to_csv(summary_path, index=False)
windows_df.to_csv(windows_path, index=False)
residual_metrics_df.to_csv(residual_metrics_path, index=False)
residual_grid_df.to_csv(residual_grid_path, index=False)
scaling_fit_df.to_csv(scaling_fit_path, index=False)

figure_paths = [
    fig1_path,
    fig2_path,
    fig3_path,
    fig4_path,
    fig5_path,
    fig6_path,
    fig7_path,
    fig8_path,
    fig9_path,
    fig10_path,
    fig11_path,
]

metadata = {
    "params": params,
    "summary": summary,
    "measurement": measurement,
    "cgcs": cgcs,
    "figures": [str(p) for p in figure_paths],
    "data": {
        "summary": str(summary_path),
        "windows": str(windows_path),
        "residual_metrics": str(residual_metrics_path),
        "residual_grid": str(residual_grid_path),
        "scaling_fit_summary": str(scaling_fit_path),
    },
    "docs": {
        "interpretation": str(interpretation_md_path),
        "design_notes": str(design_notes_md_path),
    },
    "tex": {
        "summary_snippet": str(summary_tex_path),
        "math_notes": str(math_tex_path),
    },
}

metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

figures_md = "\n\n## Figures\n\n"
for i, fig in enumerate(figure_paths, start=1):
    figures_md += f"### Figure {i} — {fig.stem.replace('_', ' ').title()}\n\n"
    figures_md += f"![Figure {i}](../figures/{fig.name})\n\n"

interpretation_md_path.write_text(interpretation + figures_md, encoding="utf-8")

design_notes = f'''
# Design Notes — {NOTEBOOK_TITLE}

## Notebook role

Notebook 14 follows Notebook 13 by fitting a scale law to residual magnitude.

## Constraint

The residual is:

$$
\\Delta(z,x)=f_{{emp}}(z,x)-e^{{-z}}.
$$

The fitted scale law is:

$$
\\|\\Delta(z,x)\\| \\sim C(\\log x)^{{-\\alpha}}.
$$

## Measurement

Metrics include residual L1, residual L2, JS divergence, positive/negative residual mass, tail bias, and fit residuals.

## CGCS score

$$
CGCS_{{scale}} =
\\frac{{1}}{{1+\\overline{{\\|\\Delta\\|_1}}+\\overline{{JS}}+\\overline{{|r_{{fit}}|}}}}.
$$

## Handoff

Notebook 15 should decompose residuals by residue class or local arithmetic structure.
'''.strip()

design_notes_md_path.write_text(design_notes + "\n", encoding="utf-8")

summary_tex = rf'''
\section*{{{NOTEBOOK_TITLE}}}

This notebook fits a residual scaling law for normalized prime-gap deviations.

\[
\Delta(z,x)=f_{{emp}}(z,x)-e^{{-z}}
\]

\[
\|\Delta(z,x)\|\sim C(\log x)^{{-\alpha}}
\]

\begin{{itemize}}
  \item Windows used: {measurement["window_count_used"]}
  \item Mean residual $L_1$: {measurement["mean_residual_l1"]:.6f}
  \item Mean residual $L_2$: {measurement["mean_residual_l2"]:.6f}
  \item Mean JS divergence: {measurement["mean_js_divergence"]:.6f}
  \item L1 alpha: {best_alpha_l1:.6f}
  \item L2 alpha: {best_alpha_l2:.6f}
  \item JS alpha: {best_alpha_js:.6f}
  \item CGCS scaling score: {cgcs_score:.6f}
\end{{itemize}}

Residual scaling is treated as a finite-scale empirical diagnostic, not an asymptotic proof.
'''.strip()

summary_tex_path.write_text(summary_tex + "\n", encoding="utf-8")

math_tex = rf'''
\documentclass{{article}}
\usepackage{{amsmath}}
\usepackage{{amssymb}}
\usepackage[margin=1in]{{geometry}}

\begin{{document}}

\section*{{Math Notes: {NOTEBOOK_TITLE}}}

\subsection*{{Normalized gap}}

\[
z_n = \frac{{p_{{n+1}}-p_n}}{{\log p_n}}
\]

\subsection*{{Exponential baseline}}

\[
f_0(z)=e^{{-z}}, \quad z\ge 0
\]

\subsection*{{Residual}}

\[
\Delta(z,x)=f_{{emp}}(z,x)-f_0(z)
\]

\subsection*{{Scaling law}}

\[
\|\Delta(z,x)\|\sim C(\log x)^{{-\alpha}}
\]

\subsection*{{Log-linear fit}}

\[
\log \|\Delta(z,x)\| = \log C - \alpha \log\log x
\]

\subsection*{{Residual CGCS score}}

\[
CGCS_{{scale}} =
\frac{{1}}{{1+\overline{{\|\Delta\|_1}}+\overline{{JS}}+\overline{{|r_{{fit}}|}}}}
\]

\subsection*{{Interpretation}}

If $\alpha>0$, the finite-scale residual decreases under the selected measurement.

\end{{document}}
'''.strip()

math_tex_path.write_text(math_tex + "\n", encoding="utf-8")

summary_path, windows_path, residual_metrics_path, residual_grid_path, scaling_fit_path, metadata_path, interpretation_md_path, design_notes_md_path, summary_tex_path, math_tex_path

## 11. Optional results bundle

This creates a root-level export zip containing figures, data, docs, and TeX outputs.

In [ ]:

EXPORT_NAME = f"{NOTEBOOK_ID}_export.zip"

with zipfile.ZipFile(EXPORT_NAME, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in [DOCS_DIR, DATA_DIR, FIG_DIR, TEX_DIR]:
        for path in folder.rglob("*"):
            if path.is_file():
                z.write(path, path.as_posix())

print(f"Export ready: {EXPORT_NAME}")
print("Tip: uncomment Colab lines below to download.")

# --- Optional Colab download ---
# Uncomment the lines below when running in Colab
#
# from google.colab import files
# files.download(EXPORT_NAME)

## 12. Next notebook handoff

Next:

> Notebook 15 should decompose residuals by residue class or local arithmetic structure.

In [ ]:

next_step = "Notebook 15: residue-class residual decomposition."
print(next_step)